## 02 — Analysis

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv("../output/anes_clean.csv")
df.head()

### Descriptive stats

In [ ]:
df[['redist','income','partyid']].describe().round(2)

In [ ]:
# observations per survey year
df.groupby('year').size().rename('n')

### Income-redistribution correlation by decade

In [ ]:
def decade(y):
    return f"{int(y)//10*10}s"

df['decade'] = df['year'].apply(decade)

# expect this to be weak — that's the finding
corr_by_decade = (df.groupby('decade')
                    .apply(lambda g: g['redist'].corr(g['income']), include_groups=False)
                    .rename('r_income_redist')
                    .round(3))
corr_by_decade

### Partisan gap over time

In [ ]:
gap = (df[df['lowinc']]
       .groupby(['year','party3'])['redist']
       .mean()
       .unstack()
       .assign(gap=lambda x: x['Democrat'] - x['Republican'])
       .round(3))
gap

### OLS Regressions

In [ ]:
# baseline: does income predict redistribution views?
m1 = smf.ols("redist ~ income", data=df).fit()
print(m1.summary().tables[1])

In [ ]:
# add party ID
m2 = smf.ols("redist ~ income + partyid", data=df).fit()
print(m2.summary().tables[1])

In [ ]:
df['year_c'] = df['year'] - df['year'].mean()

# interaction term tests if partisan effect grew over time
m3 = smf.ols("redist ~ income + partyid * year_c", data=df).fit()
print(m3.summary().tables[1])

In [ ]:
pd.DataFrame({
    'model': ['income only', 'income + partyid', 'partyid x year'],
    'R2': [m1.rsquared, m2.rsquared, m3.rsquared],
    'N': [int(m1.nobs), int(m2.nobs), int(m3.nobs)]
}).round(3)